In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "telco_customer_churn_raw.csv"
CLEANED_DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "telco_customer_churn_cleaned.csv"

print("Raw file exists:", RAW_DATA_PATH.exists())
print("Cleaned data will be saved to:", CLEANED_DATA_PATH)

Raw file exists: True
Cleaned data will be saved to: v:\Customer-Churn-Analysis\data\cleaned\telco_customer_churn_cleaned.csv


In [2]:
df = pd.read_csv(RAW_DATA_PATH)

print("Raw dataset shape:", df.shape)
display(df.head())

Raw dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
text_columns = df.select_dtypes(include="object").columns.tolist()

for column in text_columns:
    df[column] = df[column].astype(str).str.strip()

print("Text values standardized.")

Text values standardized.


In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

blank_total_charges = df["TotalCharges"].isna()

print("Rows with missing TotalCharges after conversion:", blank_total_charges.sum())
print(
    "All missing TotalCharges rows have tenure = 0:",
    (df.loc[blank_total_charges, "tenure"] == 0).all()
)

display(
    df.loc[
        blank_total_charges,
        ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
    ]
)

df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

print("Missing TotalCharges after fill:", df["TotalCharges"].isna().sum())

Rows with missing TotalCharges after conversion: 11
All missing TotalCharges rows have tenure = 0: True


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


Missing TotalCharges after fill: 0


In [5]:
df["SeniorCitizenLabel"] = df["SeniorCitizen"].map({
    0: "No",
    1: "Yes"
})

df["ChurnFlag"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

tenure_bins = [-1, 0, 12, 24, 48, 72]
tenure_labels = [
    "0 months",
    "1-12 months",
    "13-24 months",
    "25-48 months",
    "49-72 months"
]

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=tenure_bins,
    labels=tenure_labels
)

display(
    df[
        [
            "customerID",
            "SeniorCitizen",
            "SeniorCitizenLabel",
            "tenure",
            "TenureGroup",
            "Churn",
            "ChurnFlag",
            "TotalCharges"
        ]
    ].head()
)

,customerID,SeniorCitizen,SeniorCitizenLabel,tenure,TenureGroup,Churn,ChurnFlag,TotalCharges
0,7590-VHVEG,0,No,1,1-12 months,No,0,29.85
1,5575-GNVDE,0,No,34,25-48 months,No,0,1889.50
2,3668-QPYBK,0,No,2,1-12 months,Yes,1,108.15
3,7795-CFOCW,0,No,45,25-48 months,No,0,1840.75
4,9237-HQITU,0,No,2,1-12 months,Yes,1,151.65


In [6]:
print("Cleaned dataset shape:", df.shape)

print("\nMissing values by column:")
display(df.isna().sum().to_frame(name="missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

print("\nChurnFlag values:")
display(df["ChurnFlag"].value_counts(dropna=False).sort_index())

print("\nSeniorCitizenLabel values:")
display(df["SeniorCitizenLabel"].value_counts(dropna=False))

print("\nTenureGroup values:")
display(df["TenureGroup"].value_counts(dropna=False).sort_index())

print("\nTotalCharges data type:", df["TotalCharges"].dtype)
print("Negative TotalCharges:", (df["TotalCharges"] < 0).sum())

Cleaned dataset shape: (7043, 24)

Missing values by column:


,missing_values
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0



Duplicate rows: 0
Duplicate customer IDs: 0

ChurnFlag values:


ChurnFlag
0    5174
1    1869
Name: count, dtype: int64


SeniorCitizenLabel values:


SeniorCitizenLabel
No     5901
Yes    1142
Name: count, dtype: int64


TenureGroup values:


TenureGroup
0 months          11
1-12 months     2175
13-24 months    1024
25-48 months    1594
49-72 months    2239
Name: count, dtype: int64


TotalCharges data type: float64
Negative TotalCharges: 0


In [7]:
columns_to_validate = [
    "gender",
    "SeniorCitizenLabel",
    "Partner",
    "Dependents",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "Churn"
]

for column in columns_to_validate:
    print(f"\n--- {column} ---")
    display(
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )


--- gender ---


,gender,count
0,Male,3555
1,Female,3488



--- SeniorCitizenLabel ---


,SeniorCitizenLabel,count
0,No,5901
1,Yes,1142



--- Partner ---


...

,PaperlessBilling,count
0,Yes,4171
1,No,2872



--- PaymentMethod ---


,PaymentMethod,count
0,Electronic check,2365
1,Mailed check,1612
2,Bank transfer (automatic),1544
3,Credit card (automatic),1522



--- Churn ---


,Churn,count
0,No,5174
1,Yes,1869


In [11]:
cleaning_log = """# Data Cleaning Log

## Source
- Raw file: `data/raw/telco_customer_churn_raw.csv`
- Raw dataset shape: 7,043 rows and 21 columns

## Cleaning steps
1. Preserved the raw dataset without modification.
2. Removed leading and trailing whitespace from text columns.
3. Converted `TotalCharges` from text to numeric.
4. Identified 11 blank `TotalCharges` values.
5. Verified all 11 blank `TotalCharges` records had `tenure = 0`.
6. Replaced those 11 missing `TotalCharges` values with `0.0`.
7. Added `SeniorCitizenLabel` with values `Yes` and `No`.
8. Added `ChurnFlag` where `Yes = 1` and `No = 0`.
9. Added `TenureGroup` for lifecycle-based churn analysis.
10. Confirmed no duplicate rows, duplicate customer IDs, missing values, or negative total charges remained.

## Output
- Cleaned file: `data/cleaned/telco_customer_churn_cleaned.csv`
- Cleaned dataset shape: 7,043 rows and 24 columns
"""

CLEANING_LOG_PATH = PROJECT_ROOT / "docs" / "data_cleaning_log.md"

with open(CLEANING_LOG_PATH, "w", encoding="utf-8") as file:
    file.write(cleaning_log)

print("Cleaning log saved:", CLEANING_LOG_PATH)
print("File exists:", CLEANING_LOG_PATH.exists())

Cleaning log saved: v:\Customer-Churn-Analysis\docs\data_cleaning_log.md
File exists: True


In [9]:
CLEANED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(CLEANED_DATA_PATH, index=False)

print("Cleaned dataset saved successfully.")
print("Saved file:", CLEANED_DATA_PATH)
print("File exists:", CLEANED_DATA_PATH.exists())
print("File size in bytes:", CLEANED_DATA_PATH.stat().st_size)

Cleaned dataset saved successfully.
Saved file: v:\Customer-Churn-Analysis\data\cleaned\telco_customer_churn_cleaned.csv
File exists: True
File size in bytes: 1104693
